In [1]:
%pip install -Uq opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
print(cv2.__version__)

4.13.0


In [3]:
import cv2

# 이미지 읽기
image = cv2.imread('image.png')

In [4]:
# 이미지 표시
cv2.imshow('Image', image)
cv2.waitKey(0)  # 키 입력이 있을 때까지 대기
cv2.destroyAllWindows() # 열린 모든 창 닫기

In [5]:
# 그레이스케일 변환
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

cv2.imshow('Gray Image', gray_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [6]:
# 이미지 크기 조절
width, height = 512, 512
resized_image = cv2.resize(image, (width, height))

cv2.imshow('Resized image', resized_image )
cv2.waitKey(0)
cv2.destroyAllWindows()

In [7]:
# 이미지 회전 (시계 방향 90도)
rotated_image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)

cv2.imshow('Rotated image', rotated_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [8]:
# 가우시안 블러
blurred_image = cv2.GaussianBlur(image, (15, 15), 0)

cv2.imshow('Blurred image', blurred_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
# Canny 에지 검출(임계값 하한선/상한선)
edges = cv2.Canny(image, threshold1=100, threshold2=200)

cv2.imshow('Edges', edges)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
# 이진화 처리
ret, binary_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)

cv2.imshow('Binary Image', binary_image )
cv2.waitKey(0)
cv2.destroyAllWindows()

In [11]:
# 얼굴 탐지기 로드
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

In [12]:
# 이미지 읽기 및 그레이스케일 변환
image = cv2.imread('face.jpg')
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# 얼굴 탐지
faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=5)

# 탐지된 얼굴에 바운딩 박스 그리기
for (x, y, w, h) in faces:
    cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# 결과 이미지 표시
cv2.imshow('Face Detection', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [14]:
# HOG 디스크립터 초기화
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# 이미지 읽기
image = cv2.imread('pedestrians.jpg')

# 보행자 검출
(rects, weights) = hog.detectMultiScale(image, winStride=(4, 4),
                                        padding=(8, 8), scale=1.05)

# 검출된 바운딩 박스 그리기
for (x, y, w, h) in rects:
    cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# 결과 이미지 표시
cv2.namedWindow('Pedestrian Detection', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Pedestrian Detection', 1000, 700)
cv2.imshow('Pedestrian Detection', image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [15]:
# 동영상 파일에서 영상 받아오기
cap = cv2.VideoCapture('video.mp4')

# 웹캠에서 영상 받아오기
# cap = cv2.VideoCapture(0)

In [ ]:
while cap.isOpened():       # 비디오 캡쳐 객체가 열려있는 동안 반복
    ret, frame = cap.read() # 다음 프레임 읽기 (성공여부, 이미지 반환)
    if not ret:
        break

    # 프레임 처리 (예: 그레이스케일 변환)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 결과 프레임 표시
    cv2.imshow('Video', gray_frame)

    # 'q' 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 자원 해제
cap.release()
cv2.destroyAllWindows()

In [ ]:
# 웹캠 객체탐지
import cv2
import numpy as np
from ultralytics import YOLO

# YOLO 모델 로드
model = YOLO('yolo11n.pt')

# 클래스 라벨별 색상 설정 함수
def get_colors(num_colors):
    np.random.seed(0)
    return [tuple(np.random.randint(0, 255, 3).tolist()) for _ in range(num_colors)]


# 클래스 라벨 및 색상 설정
class_names = model.names
num_classes = len(class_names)
colors = get_colors(num_classes)
print(class_names)

def detect_objects(image: np.array):
    """
    카메라에서 프레임이 전송될때 이미 ndarray이므로 PIL Image로 변환하지 않음
    """
    results = model(image, verbose=False)  # 객체 탐지
    class_names = model.names  # 클래스 이름 저장

    # 결과를 바운딩 박스와 정확도로 이미지에 표시
    for result in results:
        boxes = result.boxes.xyxy  # 바운딩 박스
        confs = result.boxes.conf  # 신뢰도
        cls = result.boxes.cls  # 클래스
        for box, confidence, class_id in zip(boxes, confs, cls):
            x1, y1, x2, y2 = map(int, box)  # 좌표를 정수로 변환
            label = class_names[int(class_id)]  # 클래스 이름
            cv2.rectangle(image, (x1, y1), (x2, y2), colors[int(class_id)], 2)
            cv2.putText(image, f'{label} {confidence:.2f}', (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    return image


# 카메라에서 프레임 캡처
# cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) # Window전용 DirectShow 백엔드 사용
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('카메라를 찾을 수 없습니다.')

while cap.isOpened():
    ret, frame = cap.read()
    # print(ret, frame)
    if not ret:
        break

    result_image = detect_objects(frame)    # 처리된 프레임을 'Video' 창에 출력

    # 프레임을 화면에 표시
    cv2.imshow('Frame', np.array(result_image))

    # 'q' 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 리소스 해제
cap.release()  # 비디오 캡처 장치/파일 해제
cv2.destroyAllWindows() # 열린 모든 창 닫기
cv2.waitKey(1) # mac은 창이 자동으로 닫히지 않아서 이 코드 추가

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Playdata\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl